# Ministral-8B-Instruct-2410 Inference — NL to ASP Translation

## 0 · Login & Imports

In [ ]:
from huggingface_hub import login
login('YOUR HUGGINGFACE_TOKEN')


In [ ]:
import torch
import json
import os
from pathlib import Path
from transformers import AutoTokenizer, pipeline
from peft import AutoPeftModelForCausalLM
from tqdm import tqdm
import pandas as pd
from datasets import load_dataset

os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("Libraries loaded.")
print(f"torch       : {torch.__version__}")
print(f"CUDA avail  : {torch.cuda.is_available()}")
print(f"GPU         : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

## 1 · Load Fine-Tuned Adapter

In [ ]:
# ── Update this path to your saved adapter / best checkpoint ──────────────────
adapter_path = "Path/To/Your/Saved/Adapter"


model = AutoPeftModelForCausalLM.from_pretrained(
    adapter_path,
    device_map="auto",
    torch_dtype=torch.bfloat16,   # Ministral training precision — bfloat16
)
tokenizer = AutoTokenizer.from_pretrained(adapter_path)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Model loaded from : {adapter_path}")
print(f"Model class       : {model.__class__.__name__}")
device_map = getattr(model, 'hf_device_map', None) or getattr(model.base_model, 'hf_device_map', 'N/A')
print(f"Device map        : {device_map}")
print(f"GPU memory        : {torch.cuda.memory_allocated() / 1e9:.2f} GB")


## 2 · Load Test Dataset

In [ ]:
SYSTEM_PROMPT = (
    "You are an expert in Translating the Natural language (NL) into "
    "Answer Set Programming (ASP) translation. "
    "Always provide precise, syntactically and semantically correct translations of NL into ASP."
)

def load_data(path, test_size=0.0, seed=42):
    """
    Load JSON dataset. Dataset and convert to conversational format for NL → ASP.
    Message format:
        user: "{SYSTEM_PROMPT}\n\nTranslate the following...: {NL}"
        assistant: "{ASP}"
    """
    def create_conversation(sample):
        return {
            "messages": [
                {
                    "role": "user",
                    "content": (
                        f"{SYSTEM_PROMPT}\n\n"
                        f"Translate the following natural language to Answer Set Programming:\n\n"
                        f"natural language: {sample['NL_V2']}"
                    )
                },
                {
                    "role": "assistant",
                    "content": sample["ASP"]
                },
            ]
        }

    dataset = load_dataset("json", data_files=path, field="data_dict", split="train")
    dataset = dataset.map(create_conversation, remove_columns=dataset.features, batched=False)

    if test_size == 0:
        return dataset, None

    split = dataset.train_test_split(test_size=test_size, seed=seed)
    return split["train"].shuffle(seed=seed), split["test"]


In [ ]:
dataset_file = "path/to/your/test_dataset.json"
testset, _ = load_data(dataset_file, test_size=0)

print(f"Test dataset size: {len(testset)}")
print("\nExample from test set:")
print(testset[0])


## 3 · Decoder Class


In [ ]:
import torch
from transformers import pipeline, PreTrainedTokenizer
from abc import ABC, abstractmethod

os.environ["TOKENIZERS_PARALLELISM"] = "false"


# ─────────────────────────────────────────────────────────────────────────────
# Abstract base
# ─────────────────────────────────────────────────────────────────────────────
class CNLDecoder(ABC):
    @abstractmethod
    def decode(self, prompt: str, max_new_tokens=100, temperature=0.5) -> str:
        pass


# ─────────────────────────────────────────────────────────────────────────────
# Naive (unconstrained) decoder
# No strip_think_tokens — Ministral has no thinking mode
# ─────────────────────────────────────────────────────────────────────────────
class NaiveCNLDecoder(CNLDecoder):

    def __init__(self, model, tokenizer: PreTrainedTokenizer):
        self.tokenizer = tokenizer
        self.model     = model
        self.device    = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
        self.pipeline  = pipeline(
            "text-generation",
            model=self.model,
            tokenizer=self.tokenizer,
            device_map=self.device
        )

    def decode(self, prompt: str, max_new_tokens=512, temperature=0) -> str:
        outputs = self.pipeline(
            prompt,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=temperature,
            top_p=None,
            eos_token_id=self.tokenizer.eos_token_id,
            pad_token_id=self.tokenizer.pad_token_id,
        )
        generated_text = outputs[0]['generated_text']
        prediction     = generated_text[len(prompt):].strip()

        # Remove EOS token if present
        if self.tokenizer.eos_token and self.tokenizer.eos_token in prediction:
            prediction = prediction.split(self.tokenizer.eos_token)[0].strip()

        # Ministral: no think tokens — no strip_think_tokens needed
        return prediction


print("✓ Decoder classes defined.")


## 4 · SyntaxTester (with checkpoint saving)

`__predict` builds the prompt with system content prepended into the user message —
no `system` role (tokenizer drops it). `__syntax_check` validates ASP output via the compile API.


In [ ]:
import requests
import json
import os
from tqdm import tqdm

COMPILE_URL = "http://YOUR_SERVER/api/compile"
API_KEY     = "YOUR_API_KEY_HERE"
HEADERS     = {"Content-Type": "application/json", "X-API-KEY": API_KEY}


class SyntaxTester:
    def __init__(self, decoder: CNLDecoder, tokenizer: PreTrainedTokenizer, test_dataset,
                 checkpoint_path="eval_checkpoint_ministral8B.json"):
        self.test_dataset    = test_dataset
        self.decoder         = decoder
        self.tokenizer       = tokenizer
        self.checkpoint_path = checkpoint_path

        if os.path.exists(self.checkpoint_path):
            with open(self.checkpoint_path, 'r') as f:
                checkpoint = json.load(f)
                self.error_count   = checkpoint.get("error_count", 0)
                self.wrong_indexes = checkpoint.get("wrong_indexes", [])
                self.predictions   = checkpoint.get("predictions", [])
                print(f"Resuming from sample {len(self.predictions)}...")
        else:
            self.error_count   = 0
            self.wrong_indexes = []
            self.predictions   = []

    def evaluate(self, verbose=False, save_every=10):
        start_idx = len(self.predictions)

        for idx in tqdm(range(start_idx, len(self.test_dataset)), desc="Evaluating"):
            sample   = self.test_dataset[idx]
            # User content contains: SYSTEM_PROMPT + \n\n + instruction + NL
            # Strip the instruction prefix to get the raw NL input
            user_content = sample["messages"][0]["content"]
            input_nl = user_content.split(
                "Translate the following natural language to Answer Set Programming:\n\nnatural language: "
            )[-1].strip()

            predicted_asp = self.__predict(input_nl)

            if verbose:
                print(f"\n[{idx+1}/{len(self.test_dataset)}] NL: {input_nl}")
                print(f"  Predicted ASP: {predicted_asp}")

            if not self.__syntax_check(predicted_asp):
                self.error_count += 1
                self.wrong_indexes.append(idx)

            self.predictions.append({
                "input_nl":      input_nl,
                "predicted_asp": predicted_asp
            })

            if (idx + 1) % save_every == 0 or (idx + 1) == len(self.test_dataset):
                self._save_progress()

        total_samples = len(self.test_dataset)
        accuracy = (total_samples - self.error_count) / total_samples if total_samples > 0 else 0.0

        if verbose:
            print(f"\nTotal samples : {total_samples}")
            print(f"ASP errors    : {self.error_count}")
            print(f"Accuracy      : {accuracy:.4f}")

        return accuracy

    def _save_progress(self):
        checkpoint = {
            "error_count":   self.error_count,
            "wrong_indexes": self.wrong_indexes,
            "predictions":   self.predictions
        }
        with open(self.checkpoint_path, 'w') as f:
            json.dump(checkpoint, f)

    def __predict(self, input_text: str, max_new_tokens=512) -> str:
        """
        Build inference prompt matching exactly the training format:
        <s>[INST]{SYSTEM_PROMPT}\n\nTranslate the following...: {input}[/INST]

        IMPORTANT: use 'user' role only — no 'system' role.
        The tokenizer silently drops system messages.
        """
        prompt = self.tokenizer.apply_chat_template(
            [
                {
                    "role": "user",
                    "content": (
                        f"{SYSTEM_PROMPT}\n\n"
                        f"Translate the following natural language to Answer Set Programming:\n\n"
                        f"natural language: {input_text}"
                    )
                }
            ],
            tokenize=False,
            add_generation_prompt=True,
        )
        return self.decoder.decode(prompt, max_new_tokens=max_new_tokens, temperature=0.1)

    def __syntax_check(self, asp_text: str) -> bool:
        """
        Validate generated ASP by attempting compilation via the API.
        Returns True if compilation succeeds (non-empty ASP returned).
        """
        try:
            response = requests.post(
                COMPILE_URL,
                headers=HEADERS,
                json={"asp": asp_text},
                timeout=10
            )
            response.raise_for_status()
            result = response.json()
            # Consider valid if no error key and a result is returned
            return "error" not in result and bool(result)
        except (requests.exceptions.RequestException, json.JSONDecodeError):
            return False


print("SyntaxTester (NL→ASP) defined.")


## 5 · Single NL Test

In [ ]:
single_NL = "Give Your Test NL Here."

decoder: CNLDecoder = NaiveCNLDecoder(model=model, tokenizer=tokenizer)

tester = SyntaxTester(
    decoder=decoder,
    tokenizer=tokenizer,
    test_dataset=[{
        "messages": [
            {
                "role": "user",
                "content": (
                    f"{SYSTEM_PROMPT}\n\n"
                    f"Translate the following natural language to Answer Set Programming:\n\n"
                    f"natural language: {single_NL}"
                )
            }
        ]
    }]
)

results = tester.evaluate(verbose=True)
print(f"\nAccuracy: {results}")
wrong_sentences = [tester.predictions[i] for i in tester.wrong_indexes]
print("Predictions:", tester.predictions)
wrong_sentences


## 6 · ASPGenerator

In [ ]:

class ASPGenerator(SyntaxTester):
    def __init__(self, decoder: CNLDecoder, tokenizer: PreTrainedTokenizer, test_dataset):
        super().__init__(decoder, tokenizer, test_dataset)
        self.data_dict          = []
        self.compilation_errors = 0

    def __validate_asp(self, asp_text: str) -> bool:
        """Validate ASP output via the compile API."""
        try:
            response = requests.post(
                COMPILE_URL, headers=HEADERS, json={"asp": asp_text}, timeout=10
            )
            result = response.json()
            return "error" not in result and bool(result)
        except Exception:
            return False

    def process_and_save(
        self,
        json_path: str,
        output_filename: str = "path/to/your/output_file.csv",
        verbose: bool = True
    ):
        with open(json_path, 'r') as f:
            raw_data = json.load(f)

        samples = raw_data.get("data_dict", [])
        self.data_dict          = []
        self.compilation_errors = 0
        total_samples           = len(samples)

        print(f"Starting Ministral-8B NL→ASP Pipeline: processing {total_samples} samples...")

        for item in tqdm(samples, disable=not verbose):
            nl_input   = item.get('NL_V2', '')
            actual_asp = item.get('ASP', '')
            category   = item.get('Category', 'N/A')
            item_id    = item.get('ID', item.get('Id', 'N/A'))

            # Direct NL → ASP inference (no CNL intermediate)
            predicted_asp   = self._SyntaxTester__predict(nl_input)
            is_valid_syntax = self.__validate_asp(predicted_asp)

            if not is_valid_syntax:
                self.compilation_errors += 1

            self.data_dict.append({
                'Natural Language': nl_input,
                'Predicted ASP':    predicted_asp,
                'Actual ASP':       actual_asp,
                'ASP Valid':        is_valid_syntax,
                'Category':         category,
                'ID':               item_id
            })

        results_df = pd.DataFrame(self.data_dict)
        results_df.to_csv(output_filename, index=False)

        if verbose:
            asp_acc = (results_df['ASP Valid'].sum() / total_samples) * 100
            print("\n" + "="*40)
            print("MINISTRAL-8B NL→ASP EVALUATION SUMMARY")
            print("="*40)
            print(f"Total Samples     : {total_samples}")
            print(f"ASP Valid         : {int(results_df['ASP Valid'].sum())}")
            print(f"ASP Accuracy      : {asp_acc:.2f}%")
            print(f"ASP Failures      : {self.compilation_errors}")
            print(f"CSV saved to      : {output_filename}")
            print("="*40)

        return self.data_dict


print("ASPGenerator (NL→ASP) defined.")


## 8 · Run Full Evaluation Pipeline

In [ ]:
dataset_file = "Path/To/Your/Test_Dataset.json"

decoder: CNLDecoder = NaiveCNLDecoder(model=model, tokenizer=tokenizer)

generator = ASPGenerator(decoder=decoder, tokenizer=tokenizer, test_dataset=None)

results = generator.process_and_save(
    json_path=dataset_file,
    output_filename="path/to/your/output_file.csv"
)